# Review Collection Log

This notebook reviews the output of `02_data_collection.ipynb` to identify data quality issues — failed pulls, missing fields, and insufficient history — and decide how each affected company should be handled (ticker correction, exclusion, or acceptance with a noted limitation).

In [1]:
import sys
sys.path.append("..")

from src import config
from src.file_utils import load_file

import pandas as pd

## Identify Failed Pulls

Filter the collection log for any entries with `status == "fail"` and list the affected tickers. Each one is investigated individually below — most failures are expected to be due to incorrect, delisted, or renamed tickers.

In [2]:
collection_log = load_file(config.RAW_DATA_DIR / "collection_log.csv", index_col = None)
failures = collection_log[collection_log["status"] == "fail"]
print(failures["ticker"].unique())

<StringArray>
[]
Length: 0, dtype: str


### Removed Companies — Delisted/Acquired

The following companies failed data collection due to delisting following acquisition or take-private transactions, confirmed via manual lookup. Removed from the universe as they are no longer investable.

- **DNB** (Dun & Bradstreet) — acquired by Clearlake Capital, August 2025
- **SXS.L** (Spectris) — acquired by KKR, December 2025
- **TET.L** (Treatt) — acquired by Döhler, delisted July 2026
- **RWI.L** (Renewi) — acquired by Macquarie/BCI consortium, June 2025
- **APH.L** (Alliance Pharma) — acquired by DBAY Advisors, May 2025
- **ALPH.L** (Alpha Group) — acquired by Corpay, October 2025

### Replaced

- **MEG → ONT** — Montrose Environmental renamed to Onterris Inc.
- **FI → FISV** — Fiserv's correct yfinance ticker is FISV, not FI. Original entry was a data-entry error, not a delisting.

In [3]:
partial_fields = collection_log[
    (collection_log["status"] == "success") & 
    (collection_log["missing_required_fields"].notna()) &
    (collection_log["missing_required_fields"] != "[]")
]
print(partial_fields[["ticker", "missing_required_fields"]])

     ticker                            missing_required_fields
16    RMV.L                 ['Current Debt', 'Long Term Debt']
18    RMV.L                                   ['Gross Profit']
76    SCT.L                 ['Current Debt', 'Long Term Debt']
82    CCC.L                  ['Depreciation And Amortization']
86   BYIT.L                 ['Current Debt', 'Long Term Debt']
91    NCC.L                                   ['Total Assets']
162   DGE.L                       ['Stock Based Compensation']
167  BATS.L                       ['Stock Based Compensation']
186  FEVR.L                                 ['Long Term Debt']
191   GRG.L                                   ['Current Debt']
196  NICL.L                                 ['Long Term Debt']
244  HLMA.L                              ['sharesOutstanding']
274  CRDA.L                              ['sharesOutstanding']
323   GNS.L                           ['EBITDA', 'Net Income']
344     MSM                              ['sharesOutsta

In [4]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
fixed_partial = partial_fields[partial_fields["ticker"] == "RCP.L"]
print(fixed_partial[["ticker", "missing_required_fields"]])
pd.reset_option("display.max_colwidth")
pd.reset_option("display.max_rows")

    ticker                                        missing_required_fields
376  RCP.L                             ['Current Debt', 'Long Term Debt']
377  RCP.L  ['Depreciation And Amortization', 'Stock Based Compensation']
378  RCP.L         ['Gross Profit', 'Operating Income', 'EBIT', 'EBITDA']
379  RCP.L                                                   ['industry']


## Review: Missing Required Fields

Companies flagged with `status == "success"` but at least one missing required field, across balance sheet, cash flow, and income statement pulls. Most cases involve one or two missing fields (commonly "Current Debt," "Long Term Debt," or "Stock Based Compensation" — fields some companies don't report separately). A few companies (e.g. RCP.L) are missing several fields across multiple statements and warrant closer review — likely a smaller/less-covered company with genuinely limited data.

**Decision:** one or two missing fields are accepted as a known limitation (handled downstream by treating missing data as unavailable for that factor, rather than excluding the company). Companies missing several core fields across multiple statements are reviewed individually.

In [5]:
fundamentals_log = collection_log[collection_log["data_type"].isin(["balance_sheet", "cash_flow", "income_statement"])]
fundamentals_summary = fundamentals_log.groupby("ticker").agg(
    min_rows = ("rows", "min"),
    start_date = ("start_date", "min"),
    end_date = ("end_date", "max")
)
print(fundamentals_summary.sort_values("min_rows"))
print(fundamentals_summary.sort_values("start_date"))
print(fundamentals_summary.sort_values("end_date"))

        min_rows           start_date             end_date
ticker                                                    
SCT.L          3  2022-07-31 00:00:00  2024-07-31 00:00:00
NCC.L          3  2022-05-31 00:00:00  2025-09-30 00:00:00
ABF.L          4  2023-08-31 00:00:00  2025-08-31 00:00:00
RM.L           4  2022-11-30 00:00:00  2025-11-30 00:00:00
RKT.L          4  2022-12-31 00:00:00  2025-12-31 00:00:00
...          ...                  ...                  ...
EFX            4  2022-12-31 00:00:00  2025-12-31 00:00:00
DPLM.L         4  2022-09-30 00:00:00  2025-09-30 00:00:00
DOV            4  2022-12-31 00:00:00  2025-12-31 00:00:00
KHC            4  2022-12-31 00:00:00  2025-12-31 00:00:00
WTS            4  2022-12-31 00:00:00  2025-12-31 00:00:00

[121 rows x 3 columns]
        min_rows           start_date             end_date
ticker                                                    
FEVR.L         4  2021-12-31 00:00:00  2024-12-31 00:00:00
NCC.L          3  2022-05-31 00:

## Review: Fundamentals History Length

Checked the minimum number of usable periods, and earliest/latest dates, 
per company across balance sheet, cash flow, and income statement.

Most companies show 4 periods, consistent with the ~4-year fundamentals 
window identified during the data audit. Three companies — RSW.L, SCT.L, 
NCC.L — show only 3 periods.

Investigating one affected balance sheet showed a column being kept by 
the cleaning step despite being almost entirely empty (only 2 of 7 fields 
populated, both showing 0.0) — `dropna(how="all")` only drops fully-empty 
columns. Cleaning was tightened to `dropna(thresh=4)`, requiring at least 
half the required fields to be present for a period to be kept, rather 
than only dropping fully-empty periods.

Date ranges also vary by company due to differing fiscal year-ends (e.g. 
some companies report on a September or June year-end rather than 
December), which explains apparent inconsistencies in start/end dates 
across companies — this is expected, not a data error.

**Decision:** re-run fundamentals collection with the updated cleaning 
threshold, then re-check history length and missing-field results before 
finalizing the universe.

In [7]:
num_fields = {
    "balance_sheet": len(config.BALANCE_SHEET_REQUIRED_FIELDS),
    "cash_flow": len(config.CASH_FLOW_REQUIRED_FIELDS),
    "income_statement": len(config.INCOME_STATEMENT_REQUIRED_FIELDS)
}

partial_values = collection_log[collection_log["missing_values_pct"] > 0]
print(partial_values[["ticker", "data_type", "missing_values_pct"]].sort_values("missing_values_pct", ascending = False))

     ticker         data_type  missing_values_pct
96   TRST.L     balance_sheet           28.571429
586    JKHY     balance_sheet           17.857143
556   MER.L     balance_sheet           17.857143
381    VLTO     balance_sheet           14.285714
126    WDAY     balance_sheet           14.285714
481    TECH     balance_sheet           14.285714
116     NOW     balance_sheet           14.285714
256   ROR.L     balance_sheet           10.714286
136    SNPS     balance_sheet           10.714286
121    ADSK     balance_sheet           10.714286
411    TTEK     balance_sheet           10.714286
151     BOX     balance_sheet           10.714286
431  CTEC.L     balance_sheet           10.714286
56      FDS     balance_sheet           10.714286
36      MCO     balance_sheet           10.714286
13   AUTO.L  income_statement           10.000000
191   GRG.L     balance_sheet            8.333333
186  FEVR.L     balance_sheet            8.333333
601    HCSG     balance_sheet            8.333333


## Review: Missing Value Percentages

Calculated missing_pct (missing_values / total expected cells) per 
company/statement to assess severity more precisely than raw counts alone.

Worst cases (~25-37% missing) are concentrated in balance sheet data for 
a handful of companies (TRST.L, MER.L, WDAY, CRL, NOW, ADSK, SNPS, CTEC.L, 
ROR.L, TTEK), and income statement data for AUTO.L and ADSK. Most other 
companies show low missing percentages (under ~10%), consistent with 
occasional gaps in individual fields rather than systemic data problems.

**Decision:** companies with missing_pct above ~25% in any single 
statement are reviewed individually for exclusion or acceptance with a 
noted limitation. Lower percentages are accepted as expected data gaps, 
to be handled as NaN during factor scoring.